# 다이캐스팅 공정 데이터 기반 품질 예측 분석

## 프로젝트 개요
다이캐스팅은 융융 금속을 금형에 고압으로 주입하여 정밀한 제품을 생산하는 공정입니다.
생산 효율을 높이고 불량률을 낮추기 위해 공정변수(주조 온도, 압력, 속도 등) 및 센서 데이터를 분석하여 불량 유형을 자동 예측하는 머신러닝 모델을 개발합니다.

## 비즈니스 문제 정의
### 현재 상황
- 불량(미성형, 박리, 기공, 평탄, 개재물 등)이 발생해도 **육안 검사에 의존**하여 판정 기준의 주관성과 검사 속도의 한계로 인해 생산성이 저하됩니다.
- 불량 발생 원인을 추적하기 어려워 **공정 개선 및 문제 해결이 어렵습니다.**
- 공정 데이터와 품질 검사를 효과적으로 매핑하지 못해 **실시간 품질 관리 및 재발 방지 대책이 부족**합니다.

### 해결 필요성
- 공정변수 및 센서 데이터와 불량 유형 발생 간의 관계 파악 필요
- 불량 유형을 자동 분류하는 모델 개발로 품질 예측이 가능한 시스템 구축 필요
- 불량 발생 주요 원인을 분석하여 공정 최적화를 위한 인사이트 도출

## 분석 목표 및 KPI 설정
### 분석 목표
1. 다이캐스팅 공정에서 발생하는 다양한 불량 유형(미성형, 박리, 기공, 평탄, 개재물 등)을 자동 예측하는 머신러닝 모델을 개발
2. 공정 데이터(주조 압력, 금형 온도, 주입 속도 등)와 센서 데이터(온도, 압력, 유량, 진동 등)를 활용하여 불량 여부를 판별
4. 실시간 품질 예측 체계를 구축하여 조기 경보 시스템 도입 및 불량률 감소

### 비즈니스 KPI
- 현재 불량률 대비 불량률 10% 감소
- 불량 예측 모델의 정확도 90% 이상






In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

In [ ]:
df = pd.read_csv('../data/DieCasting_Quality_Raw_Data.csv', header=[0, 1])

print("="*60)
print("데이터셋 로드 완료!")
print("="*60)

print(df.shape)
df.info()

In [ ]:
# 데이터셋 샘플 확인
print("\n" + "="*60)
print("DieCasting 샘플 데이터")
print("="*60)
display(df.head())

In [ ]:
# 원본 데이터 기초 통계 확인
print("\n" + "="*60)
print("다이캐스팅 데이터셋 기초 통계")
print("="*60)
display(df.describe(include='all').T)

## 2. 데이터 전처리

### 2-1. 컬럼명 공백 제거

In [ ]:
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

### 2-2. 컬럼명 소문자 전환

In [ ]:
df.columns = df.columns.map(lambda x: tuple(str(level).lower() for level in x))

### 2-3. 불필요한 컬럼 제거
id 값은 데이터와 무관하다고 판단하기 때문에 제거

In [ ]:
df.drop(columns=[('process', 'id')], inplace=True)
print("="*60)
print("id 컬럼 제거 완료!")
print("="*60)

### 2-3. 중복 데이터 확인

id 값을 제외하고 모든 컬럼의 값이 동일한 중복 행 제거

In [ ]:
### 2. 중복 제거
# 모든 남아있는 컬럼을 기준으로 중복 제거
initial_row_count = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
final_row_count = len(df)

# 결과 확인
print(f"제거 전 행 개수: {initial_row_count}")
print(f"제거 후 행 개수: {final_row_count}")
print(f"삭제된 중복 행 개수: {initial_row_count - final_row_count}")

### 2-4. 결측치 확인
몇 개의 컬럼에서 결측치가 확인되지만 학습/정답 데이터 분리를 위해 우선은 그대로 유지

In [ ]:
missing_process_sensor = df.loc[:, (['process', 'sensor'], slice(None))].isna().sum()
missing_process_sensor = missing_process_sensor[missing_process_sensor > 0].reset_index()
missing_process_sensor.columns = ['', 'Columns', 'Missing Count']
display(missing_process_sensor)

### 2-5. 파생 컬럼 생성
불량 유무를 파악하기 위한 파생 컬럼 생성

In [ ]:
### 4. 새로운 파생컬럼 생성

# MultiIndex 컬럼으로 'defect_flag'가 존재하는지 확인
if ('defect_flag', 'is_defect') not in df.columns:
    defect_cols = [col for col in df.columns if col[0]=='defects']
    df[('defect_flag','is_defect')] = (df[defect_cols] == 1).any(axis=1).astype(int)
    print("="*60)
    print("파생 컬럼 생성 완료!")
    print("불량 유무 (0: 정상, 1: 불량) 분포:")
    print(df[('defect_flag','is_defect')].value_counts()) # --> 0: 정상, 1: 불량
    print("="*60)
else:
    print("="*60)
    print("파생 컬럼이 이미 존재합니다!")
    print("="*60)

print(f"DataFrame shape: {df.shape}")
display(df.head(5))

## 3. 데이터 시각화

### 3-1. 데이터의 히스토그램 확인
- 히스토그램에서 몇몇 데이터의 이봉분포 확인됨
- 상품 유형에 따른 데이터 차이가 발생하기 때문으로 확인
- 상품 유형 별로 데이터를 분리한 분석 진행이 필요하다고 판단

In [ ]:
plt.figure(figsize=(30,30))
plt.subplots_adjust(hspace=0.38)
# 각 변수의 막대그래프 개수
for index,value in enumerate(df):
 sub=plt.subplot(12,5,index+1)
 sub.hist(df[value],facecolor=(144/255,171/255,221/255),
	 	 	 linewidth=.3,edgecolor='black')
 plt.title(value)

In [ ]:
# product_type 별 히스토그램 시각화
product_type_1 = df[df[('process', 'product_type')] == 1]
product_type_2 = df[df[('process', 'product_type')] == 2]

num_cols = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
num_total = len(num_cols)

n_cols_per_row = 3
n_rows = (num_total + n_cols_per_row - 1) // n_cols_per_row

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols_per_row, figsize=(n_cols_per_row * 6, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(product_type_1[col].dropna(), bins=50, alpha=0.6, edgecolor='black', linewidth=0.3, label='type 1')
    axes[i].hist(product_type_2[col].dropna(), bins=50, alpha=0.6, edgecolor='black', linewidth=0.3, label='type 2')
    axes[i].set_title(f'{col[0]}_{col[1]}')
    axes[i].legend()

for j in range(num_total, len(axes)):
    fig.delaxes(axes[j])

plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
# process, sensor 컬럼 박스플롯 (product_type 1, 2 비교)
product_type_1 = df[df[('process', 'product_type')] == 1]
product_type_2 = df[df[('process', 'product_type')] == 2]

ps_cols = [(lvl0, lvl1) for lvl0, lvl1 in df.columns if lvl0 in ['process', 'sensor']]
num_total = len(ps_cols)

n_cols_per_row = 3
n_rows = (num_total + n_cols_per_row - 1) // n_cols_per_row

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols_per_row, figsize=(n_cols_per_row * 6, n_rows * 4))
axes = axes.flatten()

for i, (lvl0, lvl1) in enumerate(ps_cols):
    data = [product_type_1[(lvl0, lvl1)].dropna(), product_type_2[(lvl0, lvl1)].dropna()]
    bp = axes[i].boxplot(data, labels=['type 1', 'type 2'], patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('coral')
    axes[i].set_title(f'{lvl0}_{lvl1}')
    axes[i].set_ylabel('값')
    axes[i].grid(True, alpha=0.3)

for j in range(num_total, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 3. 이상치 확인 및 처리


### 3-1. 불량 유형 여부 이상치
불량 유형에 해당되는 1(불량), 0(정상) 값을 벗어나는 2 또는 3과 같은 값을 1(불량) 값으로 대체




In [ ]:
### 3. 2,3을 불량 1로 대체
defect_cols = df['defects'].columns

# 'Defects' 대분류 아래의 모든 컬럼에 대해 2 이상의 값을 1로 변경
for col in defect_cols:
    # Defects 하위의 col 값 중 2 이상인 경우를 1로 치환
    df.loc[df[('defects', col)] >= 2, ('defects', col)] = 1

print("="*60)
print("이상치 값 대체 완료!")
print("="*60)

In [ ]:
# product_type 별 데이터 분리
product_type_1 = df[df[('process', 'product_type')] == 1]
product_type_2 = df[df[('process', 'product_type')] == 2]

print("="*60)
print("각 타입별로 0밖에 없는 defect 컬럼 제거 전:")
print("="*60)
print(f"type1: {product_type_1.shape}")
print(f"type2: {product_type_2.shape}")

# 각 타입별로 0밖에 없는 defect 컬럼 출력
for name, df_type in [('type1', product_type_1), ('type2', product_type_2)]:
    defect_cols = df_type['defects'].columns
    zero_cols = [col for col in defect_cols if df_type[('defects', col)].sum() == 0]
    print(f"=== {name} - 0밖에 없는 불량 컬럼 ===")
    if zero_cols:
        for col in zero_cols:
            print(f"  - {col}")
    else:
        print("  없음")
    print()

# 각 타입별로 0밖에 없는 defect 컬럼 제거
def drop_zero_defect_cols(df_type):
    defect_cols = df_type['defects'].columns
    zero_cols = [col for col in defect_cols if df_type[('defects', col)].sum() == 0]
    print(f"제거 컬럼: {zero_cols}")
    return df_type.drop(columns=[('defects', c) for c in zero_cols])

product_type_1 = drop_zero_defect_cols(product_type_1)
product_type_2 = drop_zero_defect_cols(product_type_2)

print("="*60)
print("각 타입별로 0밖에 없는 defect 컬럼 제거 후:")
print("="*60)
print(f"type1: {product_type_1.shape}")
print(f"type2: {product_type_2.shape}")

In [ ]:
# 상품 유형 컬럼 제거
product_type_1 = product_type_1.drop(columns=[('process', 'product_type')])
product_type_2 = product_type_2.drop(columns=[('process', 'product_type')])
print("="*60)
print("product_type 컬럼 제거 완료!")
print("="*60)

print("제거 후:")
print(f"type1: {product_type_1.shape}")
print(f"type2: {product_type_2.shape}")

# 전처리 완료된 파일 상품 유형 별 저장
product_type_1.to_csv('../data_processed/product_type_1.csv', index=False)
product_type_2.to_csv('../data_processed/product_type_2.csv', index=False)

print("="*60)
print("파일 저장 완료!:")
print("="*60)

In [ ]:
df_1 = pd.read_csv('../data/product_type_1.csv', header=[0, 1])
df_2 = pd.read_csv('../data/product_type_2.csv', header=[0, 1])

print(f"type1: {df_1.shape}")
print(f"type2: {df_2.shape}")